# Label Propagation Community Detection
---

## Introduction

Community detection is an unsupervised learning task used to find groups of nodes in a graph. A **community** is a set of nodes that are more strongly connected to each other than to the rest of the network.

In this notebook, we implement and test a from-scratch **Label Propagation Community Detection** algorithm. The algorithm works directly on an adjacency matrix, where each row and column represents a node and each entry represents the edge weight between two nodes.

The main goal of this notebook is to show how the written algorithm works in practice: first with a small graph, then with visualizations and interpretation of the communities it finds.


## Mathematical Intuition

Label propagation begins by assigning each node its own label. During each iteration, every node looks at the labels of its neighbors and adopts the label with the strongest support.

For an unweighted graph, this means a node takes the most common neighbor label. For a weighted graph, the vote is weighted by edge strength.

For node $i$, a weighted label score can be written as:

$$
score_i(\ell) = \sum_{j \in N(i)} A_{ij} \cdot \mathbf{1}(label_j = \ell)
$$

The node then chooses the label with the highest score:

$$
label_i = rg\max_{\ell} score_i(\ell)
$$

Where:

- $N(i)$ is the set of neighbors of node $i$
- $A_{ij}$ is the edge weight between node $i$ and node $j$
- $\ell$ is a possible community label

The process repeats until labels stop changing or the maximum number of iterations is reached.


## Import Libraries

We will use:

- `numpy` for matrices and numerical work
- `pandas` for organizing results in tables
- `matplotlib` for graph and matrix visualizations

The model itself is written from scratch and does not depend on `networkx` or `scikit-learn`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Save and Import the Written Algorithm

The next cell writes the Label Propagation Community Detection class into a local Python file. This mirrors the way a user would normally store the algorithm in a `.py` file and then import it into a notebook.


In [ ]:
%%writefile label_propagation_community_detection.py
"""
label_propagation_community_detection.py

A from-scratch implementation of Label Propagation for community detection
using NumPy.

Label propagation is an unsupervised graph algorithm used to find communities
inside a network. A community is a group of nodes that are more closely
connected to each other than to the rest of the graph.

The basic idea is simple:

1. Give every node its own unique label.
2. Visit each node and look at the labels of its neighbors.
3. Change the node's label to the label most common among its neighbors.
4. Repeat this process until labels stop changing or the maximum number of
   iterations is reached.

This implementation supports weighted adjacency matrices. That means stronger
edges can have more influence during the label vote.

This file provides:

- label_propagation_community_detection:
    A simple label propagation class for graph community detection.

- fit(X):
    Runs label propagation on an adjacency matrix.

- fit_predict(X):
    Fits the model and returns the final community labels.

The model stores:

- labels_:
    The final community label assigned to each node.

- n_iter_:
    The number of update rounds completed.

- n_communities_:
    The number of communities found.

This implementation is intended for educational use and does not rely on
networkx or scikit-learn.
"""

import numpy as np


class label_propagation_community_detection:
    def __init__(self, max_iter=100, random_state=None):
        """
        Create a label propagation community detection model.

        Parameters
        ----------
        max_iter : int
            Maximum number of full update rounds through the graph.

        random_state : int or None
            Controls random update order and tie-breaking so results can be
            reproduced.
        """
        if max_iter < 1:
            raise ValueError("max_iter must be at least 1.")

        self.max_iter = max_iter
        self.random_state = random_state

        self.labels_ = None
        self.n_iter_ = 0
        self.n_communities_ = 0

    # ---------------------------------------------------------
    # Data preparation
    # ---------------------------------------------------------

    def _prepare_adjacency_matrix(self, X):
        """
        Convert input into a valid square adjacency matrix.

        Parameters
        ----------
        X : array-like of shape (n_nodes, n_nodes)
            Adjacency matrix representing a graph.

            X[i, j] is the edge weight between node i and node j.
            A value of 0 means there is no edge.
        """
        X = np.asarray(X, dtype=float)

        if X.ndim != 2:
            raise ValueError("X must be a 2D adjacency matrix.")

        if X.shape[0] == 0:
            raise ValueError("X must contain at least one node.")

        if X.shape[0] != X.shape[1]:
            raise ValueError("X must be square with shape (n_nodes, n_nodes).")

        if np.any(X < 0):
            raise ValueError("Adjacency matrix edge weights must be nonnegative.")

        return X

    # ---------------------------------------------------------
    # Neighbor voting
    # ---------------------------------------------------------

    def _neighbor_indices(self, adjacency_matrix, node_index):
        """
        Return the indices of all nodes connected to a given node.
        """
        return np.where(adjacency_matrix[node_index] > 0)[0]

    def _weighted_label_vote(self, adjacency_matrix, labels, node_index, rng):
        """
        Choose a new label for one node based on its neighbors.

        Each neighbor votes using its current label. If the graph is weighted,
        the neighbor's edge weight determines the strength of the vote.

        If there is a tie, one of the tied labels is chosen randomly.
        """
        neighbors = self._neighbor_indices(adjacency_matrix, node_index)

        # Isolated nodes keep their current label.
        if len(neighbors) == 0:
            return labels[node_index]

        vote_totals = {}

        for neighbor in neighbors:
            neighbor_label = labels[neighbor]
            edge_weight = adjacency_matrix[node_index, neighbor]

            if neighbor_label not in vote_totals:
                vote_totals[neighbor_label] = 0.0

            vote_totals[neighbor_label] += edge_weight

        best_score = max(vote_totals.values())

        best_labels = [
            label
            for label, score in vote_totals.items()
            if score == best_score
        ]

        if len(best_labels) == 1:
            return best_labels[0]

        return rng.choice(best_labels)

    # ---------------------------------------------------------
    # Label cleanup
    # ---------------------------------------------------------

    def _compress_labels(self, labels):
        """
        Convert final labels into clean consecutive labels.

        Example:
        [4, 4, 9, 9, 9] becomes [0, 0, 1, 1, 1].
        """
        unique_labels = np.unique(labels)

        label_map = {
            old_label: new_label
            for new_label, old_label in enumerate(unique_labels)
        }

        compressed = np.array([
            label_map[label]
            for label in labels
        ])

        return compressed

    # ---------------------------------------------------------
    # Model fitting
    # ---------------------------------------------------------

    def fit(self, X):
        """
        Run label propagation on a graph.

        Parameters
        ----------
        X : array-like of shape (n_nodes, n_nodes)
            A square adjacency matrix.

            Each row and column represents a node.
            X[i, j] gives the edge weight between node i and node j.
            If X[i, j] is 0, there is no edge from node i to node j.

        Returns
        -------
        self
            The fitted community detection model.
        """
        adjacency_matrix = self._prepare_adjacency_matrix(X)

        n_nodes = adjacency_matrix.shape[0]
        rng = np.random.default_rng(self.random_state)

        # At the beginning, every node starts in its own community.
        labels = np.arange(n_nodes)

        for iteration in range(self.max_iter):
            old_labels = labels.copy()

            # Random update order helps avoid repeated update cycles.
            update_order = rng.permutation(n_nodes)

            for node_index in update_order:
                labels[node_index] = self._weighted_label_vote(
                    adjacency_matrix,
                    labels,
                    node_index,
                    rng
                )

            self.n_iter_ = iteration + 1

            # Stop early if a full pass causes no label changes.
            if np.array_equal(labels, old_labels):
                break

        self.labels_ = self._compress_labels(labels)
        self.n_communities_ = len(np.unique(self.labels_))

        return self

    def fit_predict(self, X):
        """
        Fit the model and return the final community labels.

        Parameters
        ----------
        X : array-like of shape (n_nodes, n_nodes)
            Square adjacency matrix for the graph.

        Returns
        -------
        np.ndarray
            Community label for each node.
        """
        self.fit(X)

        return self.labels_

In [ ]:
from label_propagation_community_detection import label_propagation_community_detection

## Create a Small Weighted Graph

To keep the notebook self-contained, we will create a small graph manually.

The graph has:

- Nodes `0`, `1`, `2`, and `3` in one strong community
- Nodes `4`, `5`, `6`, and `7` in another strong community
- Weak bridge edges between the two groups
- Node `8` and node `9` as a small third pair/community

The algorithm will only see the adjacency matrix. It will not be told the true communities ahead of time.


In [ ]:
# Number of nodes in the graph
n_nodes = 10

# Start with an empty adjacency matrix
A = np.zeros((n_nodes, n_nodes))

# Helper function for adding undirected weighted edges
def add_edge(i, j, weight=1.0):
    A[i, j] = weight
    A[j, i] = weight

# Community 1: nodes 0, 1, 2, 3
add_edge(0, 1, 1.0)
add_edge(0, 2, 1.0)
add_edge(1, 2, 1.0)
add_edge(1, 3, 0.9)
add_edge(2, 3, 0.9)

# Community 2: nodes 4, 5, 6, 7
add_edge(4, 5, 1.0)
add_edge(4, 6, 1.0)
add_edge(5, 6, 1.0)
add_edge(5, 7, 0.9)
add_edge(6, 7, 0.9)

# Weak bridge between the two larger communities
add_edge(3, 4, 0.15)
add_edge(2, 5, 0.10)

# Small third community: nodes 8 and 9
add_edge(8, 9, 1.0)

A

## Convert the Graph to an Edge Table

Although the algorithm uses the adjacency matrix, it is often helpful to view the graph as an edge list for interpretation.


In [ ]:
edges = []
for i in range(n_nodes):
    for j in range(i + 1, n_nodes):
        if A[i, j] > 0:
            edges.append({"node_1": i, "node_2": j, "weight": A[i, j]})

edges_df = pd.DataFrame(edges)
edges_df

## Basic Graph Statistics

Before fitting the model, we can inspect the number of nodes, number of edges, and weighted degree of each node.

The weighted degree of a node is the sum of the weights of all edges connected to that node.


In [ ]:
weighted_degree = A.sum(axis=1)

stats = pd.DataFrame({
    "node": np.arange(n_nodes),
    "weighted_degree": weighted_degree,
    "number_of_neighbors": (A > 0).sum(axis=1)
})

print("Number of nodes:", n_nodes)
print("Number of edges:", len(edges_df))

stats

## Visualize the Adjacency Matrix

The adjacency matrix shows which nodes are connected. Darker or more intense values represent stronger edges.

Blocks along the diagonal usually suggest communities because nodes in the same group connect heavily to each other.


In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(A)
plt.colorbar(label="Edge weight")
plt.title("Weighted Adjacency Matrix")
plt.xlabel("Node")
plt.ylabel("Node")
plt.xticks(range(n_nodes))
plt.yticks(range(n_nodes))
plt.show()

## Fit the Label Propagation Model

Now we apply the written algorithm to the adjacency matrix.

The `random_state` parameter is used so that the random update order and tie-breaking are reproducible.


In [ ]:
model = label_propagation_community_detection(
    max_iter=100,
    random_state=2
)

labels = model.fit_predict(A)

print("Community labels:", labels)
print("Number of communities found:", model.n_communities_)
print("Iterations completed:", model.n_iter_)

## Interpret the Results

The table below shows the community label assigned to each node.

Labels are compressed into clean consecutive integers, so community labels will appear as `0`, `1`, `2`, and so on.


In [ ]:
results = pd.DataFrame({
    "node": np.arange(n_nodes),
    "community": labels,
    "weighted_degree": weighted_degree,
    "number_of_neighbors": (A > 0).sum(axis=1)
})

results.sort_values(["community", "node"])

In [ ]:
community_summary = (
    results
    .groupby("community")
    .agg(
        nodes=("node", list),
        size=("node", "count"),
        average_weighted_degree=("weighted_degree", "mean")
    )
)

community_summary

## Visualize the Detected Communities

The next plot draws the graph using manually assigned coordinates.

- Node color represents the detected community.
- Edge thickness represents edge weight.
- Weak bridge edges should appear thinner than within-community edges.


In [ ]:
positions = {
    0: (0.0, 1.0),
    1: (1.0, 1.5),
    2: (1.0, 0.5),
    3: (2.0, 1.0),
    4: (4.0, 1.0),
    5: (5.0, 1.5),
    6: (5.0, 0.5),
    7: (6.0, 1.0),
    8: (2.5, -1.0),
    9: (3.5, -1.0),
}

def plot_graph(adjacency_matrix, labels, positions):
    plt.figure(figsize=(9, 5))

    # Draw edges first
    for i in range(adjacency_matrix.shape[0]):
        for j in range(i + 1, adjacency_matrix.shape[1]):
            weight = adjacency_matrix[i, j]
            if weight > 0:
                x_values = [positions[i][0], positions[j][0]]
                y_values = [positions[i][1], positions[j][1]]
                plt.plot(
                    x_values,
                    y_values,
                    linewidth=1 + 3 * weight,
                    alpha=0.45
                )

    # Draw nodes
    x = [positions[i][0] for i in range(len(labels))]
    y = [positions[i][1] for i in range(len(labels))]

    scatter = plt.scatter(
        x,
        y,
        c=labels,
        s=650,
        edgecolors="black"
    )

    # Add node labels
    for node in range(len(labels)):
        plt.text(
            positions[node][0],
            positions[node][1],
            str(node),
            ha="center",
            va="center",
            fontsize=11,
            color="white",
            fontweight="bold"
        )

    plt.title("Detected Communities with Label Propagation")
    plt.axis("off")
    plt.colorbar(scatter, label="Detected community")
    plt.show()

plot_graph(A, labels, positions)

## Check Sensitivity to Random State

Label propagation can depend on update order, especially when there are ties. Because of that, it is useful to run the model with several random seeds.

A stable graph structure should usually produce similar communities across different seeds.


In [ ]:
seed_results = []

for seed in range(10):
    temp_model = label_propagation_community_detection(
        max_iter=100,
        random_state=seed
    )
    temp_labels = temp_model.fit_predict(A)
    seed_results.append({
        "random_state": seed,
        "labels": temp_labels.tolist(),
        "n_communities": temp_model.n_communities_,
        "n_iter": temp_model.n_iter_
    })

pd.DataFrame(seed_results)

## Experiment: Make the Bridge Stronger

The original graph has weak bridge edges between the two main groups. Now we increase those bridge weights to see whether the algorithm merges communities.

This helps show that label propagation is driven by graph connectivity and edge weight strength.


In [ ]:
A_stronger_bridge = A.copy()

# Strengthen the bridge between the two large groups
A_stronger_bridge[3, 4] = 1.0
A_stronger_bridge[4, 3] = 1.0
A_stronger_bridge[2, 5] = 0.8
A_stronger_bridge[5, 2] = 0.8

bridge_model = label_propagation_community_detection(
    max_iter=100,
    random_state=2
)

bridge_labels = bridge_model.fit_predict(A_stronger_bridge)

print("Original labels:", labels)
print("Stronger bridge labels:", bridge_labels)
print("Original number of communities:", model.n_communities_)
print("Stronger bridge number of communities:", bridge_model.n_communities_)

In [ ]:
plot_graph(A_stronger_bridge, bridge_labels, positions)

## Conclusion

This notebook implemented and tested a from-scratch Label Propagation Community Detection algorithm.

The main takeaways are:

- Label propagation is an unsupervised method for finding communities in graphs.
- The algorithm begins with every node in its own community.
- Nodes repeatedly adopt the strongest label among their neighbors.
- Weighted edges influence the voting process more strongly than weak edges.
- The algorithm does not require choosing the number of communities in advance.
- Random update order can affect results when the graph contains ties or ambiguous connections.

This makes label propagation useful for exploratory community detection, especially when the graph structure itself is the main source of information.
